# Issue #4: Keywords preprocessing
**Author:** Aaron (`aaronstrong05`) | **Branch:** `Aaron-4` | **Closes:** #4

Turns the JSON-like `keywords` string in `data/raw/keywords.csv` into one row per movie with a clean list of keyword names.

`keywords` (string) → `keywords_parsed` (list of dicts) → `keyword_names` (list of str) → `keyword_names_clean` (normalised, de-duplicated list)

Outputs go to `data/processed/` (gitignored, not committed). Each step writes a new column instead of overwriting its source. Full write-up and decisions: `reports/preprocessing/04_keywords_cleaning.md`.

## 0. Setup

In [1]:
import ast
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../../data/raw/keywords.csv")
PARSED_PATH = Path("../../data/processed/keywords_parsed.csv")
CLEAN_PATH = Path("../../data/processed/keywords_clean.csv")

## 1. Load and inspect *(task 1)*

In [2]:
df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
print()
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Fully duplicated rows (all columns identical):", df.duplicated().sum())
df.head()

Shape: (46419, 2)

id          int64
keywords      str
dtype: object

Missing values per column:
id          0
keywords    0
dtype: int64

Fully duplicated rows (all columns identical): 987


,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


**Findings:** 46,419 rows × 2 columns, no `NaN`, 987 fully duplicated rows. Movies without keywords are stored as the string `"[]"`, not as a null.

## 2. Parse the `keywords` column *(task 2)*
The data uses single quotes, so `json.loads` would fail; `ast.literal_eval` parses it safely.

In [3]:
# Look at one raw value: it is a string, not a list
sample = df["keywords"].iloc[0]
print(sample)
print(type(sample))

[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]
<class 'str'>


In [4]:
# Test the parser on that one sample before applying it everywhere
parsed_sample = ast.literal_eval(sample)
print(parsed_sample)
print(type(parsed_sample))

[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]
<class 'list'>


In [5]:
# Parse the whole column into a NEW column (raw `keywords` is kept for comparison)
df["keywords_parsed"] = df["keywords"].apply(ast.literal_eval)

# Every value should now be a list
df["keywords_parsed"].apply(type).value_counts()

keywords_parsed
<class 'list'>    46419
Name: count, dtype: int64

In [6]:
# Save the parsed data, then reload it to confirm it round-trips correctly
PARSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_json(PARSED_PATH, orient="records")

df_parsed = pd.read_json(PARSED_PATH, orient="records")
print(df_parsed.shape)
df_parsed.head()

(46419, 3)


,id,keywords,keywords_parsed
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


## 3. Extract keyword names *(task 3)*

In [7]:
df["keyword_names"] = df["keywords_parsed"].apply(
    lambda kw_list: [kw["name"] for kw in kw_list]
)
df["keyword_names"].head()

0    [jealousy, toy, boy, friendship, friends, riva...
1    [board game, disappearance, based on children'...
2    [fishing, best friend, duringcreditsstinger, o...
3    [based on novel, interracial relationship, sin...
4    [baby, midlife crisis, confidence, aging, daug...
Name: keyword_names, dtype: object

## 4. Normalise the format *(task 4)*

In [8]:
all_kw = df["keyword_names"].explode().dropna()

print("Keywords not already lowercase:  ", (all_kw != all_kw.str.lower()).sum())
print("Keywords with extra whitespace:  ", (all_kw != all_kw.str.strip()).sum())

Keywords not already lowercase:   18
Keywords with extra whitespace:   180


**Decision:** strip whitespace and lowercase. De-duplicate with `dict.fromkeys` (keeps order and stays a list, unlike `set`).

In [9]:
def clean_keywords(names):
    """Strip and lowercase each keyword, then drop repeats while keeping order."""
    cleaned = [n.strip().lower() for n in names]
    return list(dict.fromkeys(cleaned))


df["keyword_names_clean"] = df["keyword_names"].apply(clean_keywords)

changed = df["keyword_names"] != df["keyword_names_clean"]
print("Movies whose keyword list changed after cleaning:", changed.sum())

Movies whose keyword list changed after cleaning: 196


## 5. Duplicate keywords within a movie *(task 5)*
A list has an internal repeat when `len(x) != len(set(x))`. (`Series.duplicated()` compares whole rows, so it is the wrong tool here.)

In [10]:
# How many movies had repeats once case/whitespace were normalised (before de-duplication)?
had_repeats = df["keyword_names"].apply(
    lambda names: len(names) != len({n.strip().lower() for n in names})
)
print("Movies with within-list repeats before cleaning:", had_repeats.sum())

# After cleaning there must be none
still_repeats = df["keyword_names_clean"].apply(lambda names: len(names) != len(set(names)))
print("Movies with within-list repeats after cleaning: ", still_repeats.sum())
assert still_repeats.sum() == 0, "clean_keywords left duplicates inside a list"

Movies with within-list repeats before cleaning: 0
Movies with within-list repeats after cleaning:  0


## 6. Empty keyword lists *(task 6)*
**Decision: drop them.** They carry no keyword information, and the movies remain recoverable from `movies_metadata.csv` by joining on `id`.

In [11]:
def drop_empty_keywords(df, col="keyword_names_clean"):
    """Return a new DataFrame without rows whose keyword list is empty."""
    # keep rows whose list has at least one keyword
    mask = df[col].apply(len) > 0
    # reset the index so row labels run 0..n with no gaps
    return df[mask].reset_index(drop=True)


n_before = len(df)
df = drop_empty_keywords(df)
print(f"Dropped {n_before - len(df)} movies with no keywords; {len(df)} remain")

# Test: no empty lists may remain
assert df["keyword_names_clean"].apply(len).min() >= 1

Dropped 14795 movies with no keywords; 31624 remain


## 7. Movie IDs *(tasks 7, 8, 9)*
Check for missing, invalid and repeated IDs, keep `id`, and end with one row per movie.

In [12]:
print("Missing ids:        ", df["id"].isnull().sum())
print("Non-positive ids:   ", (df["id"] <= 0).sum())
print("Repeated ids:       ", df["id"].duplicated().sum())
print("Repeated (id, keywords) pairs:", df.duplicated(subset=["id", "keywords"]).sum())

Missing ids:         0
Non-positive ids:    0
Repeated ids:        532
Repeated (id, keywords) pairs: 532


Before dropping repeated IDs, confirm the repeats are exact copies (not conflicting keyword data).

In [13]:
repeated_rows = df[df["id"].duplicated(keep=False)]
distinct_per_id = repeated_rows.groupby("id")["keywords"].nunique()
n_conflicting_ids = (distinct_per_id > 1).sum()

print("Ids that appear more than once:            ", repeated_rows["id"].nunique())
print("Of those, ids with CONFLICTING keyword data:", n_conflicting_ids)

assert n_conflicting_ids == 0, (
    "Some repeated ids have different keywords: inspect them before dropping "
    "(see repeated_rows) and decide whether to merge or keep one."
)

Ids that appear more than once:             532
Of those, ids with CONFLICTING keyword data: 0


**Decision:** all repeats are exact copies, so keep the first occurrence.

In [14]:
def drop_repeat_id(df, id_col="id"):
    """Return a new DataFrame keeping only the first row for each id."""
    # True for every repeat of an id (all copies after the first)
    mask = df[id_col].duplicated(keep="first")
    # keep the rows that are NOT repeats, with a clean 0..n index
    result = df[~mask].reset_index(drop=True)
    print(f"{mask.sum()} repeated-id rows removed, {len(result)} remain")
    return result


df = drop_repeat_id(df)

532 repeated-id rows removed, 31092 remain


In [15]:
# Final integrity checks for tasks 8 and 9
assert "id" in df.columns, "movie id must be retained"
assert df["id"].is_unique, "there must be exactly one row per movie id"
assert df["keyword_names_clean"].apply(len).min() >= 1
print("OK:", len(df), "movies, one row per id, none with empty keyword lists")

OK: 31092 movies, one row per id, none with empty keyword lists


## 8. Save the cleaned data
Only `id` and `keyword_names_clean` are saved (gitignored, not committed).

In [16]:
#create new dir to store "id" and "keyword_names_clean" as .csv
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df[["id", "keyword_names_clean"]].to_csv(CLEAN_PATH, index=False)
 #stores new directory as list of dicts to final df using ast.literal_eval to maintain structure
df_clean = pd.read_csv(CLEAN_PATH, converters={"keyword_names_clean": ast.literal_eval})

#safety checks
assert df_clean["keyword_names_clean"].to_list() == df["keyword_names_clean"].to_list()
assert df_clean["id"].to_list() == df["id"].to_list()
print(df_clean.shape)
df_clean.head()


(31092, 2)


,id,keyword_names_clean
0,862,"[jealousy, toy, boy, friendship, friends, riva..."
1,8844,"[board game, disappearance, based on children'..."
2,15602,"[fishing, best friend, duringcreditsstinger, o..."
3,31357,"[based on novel, interracial relationship, sin..."
4,11862,"[baby, midlife crisis, confidence, aging, daug..."


**Result:** 31,092 movies, one row per `id`, each with at least one cleaned keyword. Details and reasoning: `reports/preprocessing/04_keywords_cleaning.md`.